In [ ]:
# TP1 : 3 Premier notebook pandas : chargement (Parquet), dimensions, types, mémoire, taux de remplissage par colonne.

# PyArrow est la bibliothèque idéale car elle accède directement aux métadonnées 
# sans charger toutes les données en mémoire (Pour éviter tout Crash de RAM).
import pyarrow.parquet as pq

parquet_file_path = "../../food.parquet"
# Chargement du Parquet : Le parcourir en mode streaming (Une lecture en flux continu) :
parquet_file = pq.ParquetFile(parquet_file_path)

# Les métadonnées d'un fichier Parquet sont un bloc d'informations situé à la fin du fichier (le footer) qui décrit toute la structure, l'organisation et le contenu des données stockées, sans qu'il soit nécessaire de lire les données elles-mêmes.
metadata_parquet_file = parquet_file.metadata

# Les dimensions du parquet :
number_of_rows = metadata_parquet_file.num_rows
number_of_columns = metadata_parquet_file.num_columns
print(f"Ce fichier comprend {number_of_rows} lignes, et {number_of_columns} colonnes.\n")

# Les types : (Avec plus de détails)
schema = parquet_file.schema
print(f"Les types de données avec tout le détail :\n{schema}")

# Taille compressée totale (sur le disque, selon les métadonnées)
compressed_size = metadata_parquet_file.serialized_size

uncompressed_bytes = sum(
    metadata_parquet_file.row_group(i).total_byte_size
    for i in range(metadata_parquet_file.num_row_groups)
)
print(f"Taille estimée en mémoire (décompressée) : {uncompressed_bytes / (1024 * 1024):.2f} Mo")


In [ ]:
import pyarrow as pa
import numpy as np

# Définir la taille maximale de chaque lot de lignes à charger en mémoire
TAILLE_BLOC = 10000

products_sold_infr = 0
filled_nutri_score = 0
filled_nutri_score_fr = 0
filled_nutri_grade = 0

# Itérer séquentiellement sur le fichier par blocs de lignes
for batch in parquet_file.iter_batches(batch_size=TAILLE_BLOC, columns=['countries_tags', 'nutriscore_grade', 'nutriscore_score', 'nutriments']):
    # Convertir le batch/bloc en Pandas pour modifier
    data_frame = batch.to_pandas()

    # 4.1 Combien de produits vendus en France ? => products_sold_infr
    filtered_data = data_frame[(data_frame["countries_tags"].notna()) & data_frame["countries_tags"].str.contains('en:france', regex=False)]
    products_sold_infr += len(filtered_data)

    # 4.2 Quelle part a un Nutri-Score renseigné ? => Que la France
    filtered_data_fr = filtered_data[filtered_data["nutriscore_score"].notna()]
    filled_nutri_score_fr += len(filtered_data_fr)

    # 4.2 Quelle part a un Nutri-Score renseigné ? => Tout le parquet,  en se basant sur nutriscore_score
    filtered_data = data_frame[data_frame["nutriscore_score"].notna()]
    filled_nutri_score += len(filtered_data)

    # 4.2 Quelle part a un Nutri-Score renseigné ? => Tout le parquet, en se basant sur nutriscore_grade
    # filtered_data = data_frame[(data_frame["nutriscore_grade"].notna()) | (data_frame["nutriscore_grade"].isin(['unknown', 'not-applicable']))]
    # filled_nutri_grade += len(filtered_data)

# print(products_sold_infr)
# print(filled_nutri_score)
# print(filled_nutri_score_fr)
# print(filled_nutri_grade)

# 2-La proportion des nutriscores rensignés :
# 2.1-La proportion des nutriscores rensignés 'nutriscore_grade', sans les 'None', sans les 'unknown' et 'not-applicable' (Pour tout le parquet) :
#proportion_grade_filled = (filled_nutri_grade / number_of_rows) * 100
#print(filled_nutri_grade) # 1381069
#print(f"{proportion_grade_filled:.2f}%") # 29.79%

# 2.2-La proportion des nutriscores rensignés 'nutriscore_score' (Pour tout le parquet) :
proportion_score_not_none = (filled_nutri_score / number_of_rows) * 100
print(filled_nutri_score) # 1381069
print(f"{proportion_score_not_none:.2f}%") # 29.79%

# 2.3-La proportion des nutriscores rensignés 'nutriscore_score' sans les 'None' (Pour les produits vendus en France seulement) :
proportion_score_not_none_fr = (filled_nutri_score_fr / products_sold_infr) * 100
print(filled_nutri_score_fr) # 463757
print(f"{proportion_score_not_none_fr:.2f}%") # 37.18%


4593024
1381069
29.79%
463757
37.18%


In [25]:
from tabulate import tabulate

donnees = [
    [products_sold_infr, filled_nutri_score_fr, "{:.2f}%".format(proportion_score_not_none_fr), filled_nutri_score, "{:.2f}%".format(proportion_score_not_none)]
]

headers = ["Produits vendus en France", "Total des nutriscores renseignés/France", "Proportion des nutriscores rensignés/France", "Total des nutriscores renseignés/Parquet", "Proportion des nutriscores rensignés/Parquet"]

 # --- AFFICHAGE DU TABLEAU DES DONNEES ---
# Affichage du tableau avec des bordures en grille
print(tabulate(donnees, headers=headers, tablefmt="grid"))

+-----------------------------+-------------------------------------------+-----------------------------------------------+--------------------------------------------+------------------------------------------------+
|   Produits vendus en France |   Total des nutriscores renseignés/France | Proportion des nutriscores rensignés/France   |   Total des nutriscores renseignés/Parquet | Proportion des nutriscores rensignés/Parquet   |
+=============================+===========================================+===============================================+============================================+================================================+
|                     1247336 |                                    463757 | 37.18%                                        |                                    1381069 | 29.79%                                         |
+-----------------------------+-------------------------------------------+-----------------------------------------------+-----